# Partie A - Exploration, audit, raccourcis et baselines



# TP 2 - Detection du sarcasme en arabe

## ArSarcasm-v2 : desequilibre, multi-tache et generalisation


### Plan

- **Partie A** : exploration, audit, raccourcis et baselines
- **Partie B** : PLM utilise directement
- **Partie C** : embeddings statiques et reseaux profonds
- **Partie D** : PLM gele
- **Partie E** : fine-tuning mono-tache et multi-tache
- **Partie F** : tests de generalisation F1 a F6
- **Partie G** : comparaison finale et decision
- **Questions Q1 a Q12** et livrables

## 0. Environnement et regles du TP

Le decoupage officiel train/test est conserve. Le test officiel ne sert qu'a l'evaluation finale. Toutes les decisions sont prises sur la validation.

In [1]:
# Configuration simple

RANDOM_STATE = 42

import re
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

try:
    import torch
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Peripherique :', DEVICE)
    if DEVICE.type == 'cuda':
        print('GPU :', torch.cuda.get_device_name(0))
except ImportError:
    torch = None
    DEVICE = 'cpu'
    print('PyTorch indisponible : calcul CPU.')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve
)

RESULTS = []

def add_result(name, y_true, scores, threshold=0.5, split='validation'):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = (scores >= threshold).astype(int)
    row = {
        'modele': name,
        'split': split,
        'F1 sarcastique': f1_score(y_true, pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, scores),
        'Macro-F1': f1_score(y_true, pred, average='macro', zero_division=0),
        'seuil': threshold,
    }
    RESULTS.append(row)
    return row


Peripherique : cuda
GPU : NVIDIA GeForce RTX 2050


In [2]:
BASE = 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/'
full = pd.read_csv(BASE + 'training_data.csv')
test = pd.read_csv(BASE + 'testing_data.csv')
full['id'] = range(len(full))
test['id'] = range(len(test))
print('Train officiel :', full.shape)
print('Test officiel  :', test.shape)
display(full.head())


Train officiel : (12548, 5)
Test officiel  : (3000, 5)


,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4


## A3. Pretraitement arabe prudent

On conserve les emojis, hashtags, mentions, rires, allongements et ponctuation car ils peuvent porter le sarcasme. Le pretraitement depend de la famille de modele.

## A4. Taux par dialecte et sentiment, classifieur trivial

# Partie A - Exploration, audit, raccourcis et baselines

## A1. Chargement des donnees

Le sujet impose `training_data.csv` (12 548 tweets) et `testing_data.csv` (3 000 tweets).

## A2. Audit du texte brut avant normalisation

On compte les URLs, mentions, hashtags, emojis, rires, ponctuation repetee, diacritiques, tatweel, lettres repetees, caracteres latins, chiffres indiens et doublons.

In [3]:
def audit_text(frame):
    text = frame['tweet'].fillna('').astype(str)
    return pd.Series({
        'tweets': len(frame),
        'URL': text.str.count(r'https?://|www\.').sum(),
        'mentions': text.str.count('@').sum(),
        'hashtags': text.str.count('#').sum(),
        'emojis': text.map(lambda s: sum(ord(c) > 0xFFFF for c in s)).sum(),
        'rires': text.str.count('(?i)ha+h+|\u0647{3,}|\u062e{2,}').sum(),
        'ponctuation_repetee': text.str.count('[!?\u061f\u060c.]{2,}').sum(),
        'diacritiques': text.str.count('[\u064b-\u065f]').sum(),
        'tatweel': text.str.count('\u0640').sum(),
        'lettres_repetees': text.str.count(r'(.)\1{2,}').sum(),
        'caracteres_latins': text.str.count(r'[A-Za-z]').sum(),
        'chiffres_indiens': text.str.count('[\u0660-\u0669]').sum(),
    })

display(audit_text(full).to_frame('train'))
display(audit_text(test).to_frame('test'))
print('Exemples bruts train :')
display(full[['tweet']].head(30))
print('Exemples bruts test :')
display(test[['tweet']].head(30))
print('Doublons exacts train :', full['tweet'].duplicated().sum())
print('Doublons exacts test  :', test['tweet'].duplicated().sum())
print('Doublons train/test   :', len(set(full['tweet'].dropna()) & set(test['tweet'].dropna())))


,train
tweets,12548
URL,5847
mentions,3718
hashtags,13496
emojis,3444
rires,192
ponctuation_repetee,2971
diacritiques,4057
tatweel,2548
lettres_repetees,2769


,test
tweets,3000
URL,461
mentions,26
hashtags,1376
emojis,1230
rires,121
ponctuation_repetee,1223
diacritiques,2212
tatweel,1975
lettres_repetees,1579


Exemples bruts train :


,tweet
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر..."
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb..."
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf..."
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل..."
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ..."
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......
7,#انتخبوا_البرص مشكلة
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ..."
9,▪️محافظ العاصمة تفقد سير العملية الانتخابية في...


Exemples bruts test :


,tweet
0,اخوي حانق يالغلا وشفيك معصب؟ عادي تراهم بشر يف...
1,اف مو متعوده عليهم سته https://t.co/8igFPx1i26
2,اللهم اشفِ مرضانا ومرضى المسلمين . . ♥️
3,ابشركم طلقت السات 😘.
4,مؤشر خطير: ٩٠٪ من الشخصيات البرلمانية في الكوي...
5,مكتوب للغايب يوم هيعود❤❤اللهم لك الحمد حاسه رو...
6,اختطاف الأطفال في #الجزائر..العثور على جثة مشو...
7,سقف الطموحات المستقبليه بتبقى عاليه ف وقت دراس...
8,خرجك ، من قال لك تقول رأيك يحرية ؟ شو ما سمعت ...
9,نحن في هذه الحياةإماأن نكونقدوةأوعبرة فقيمتنال...


Doublons exacts train : 61
Doublons exacts test  : 4
Doublons train/test   : 2


In [4]:
def clean_text(text):
    text = '' if pd.isna(text) else str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

full['text_clean'] = full['tweet'].map(clean_text)
test['text_clean'] = test['tweet'].map(clean_text)
train, val = train_test_split(
    full, test_size=0.10, random_state=RANDOM_STATE,
    stratify=full['sarcasm']
)
X_train, X_val = train['text_clean'], val['text_clean']
y_train = train['sarcasm'].astype(int)
y_val = val['sarcasm'].astype(int)
print(len(train), len(val), len(test))
print('Taux sarcasme train/val/test :', y_train.mean(), y_val.mean(), test['sarcasm'].astype(int).mean())


11293 1255 3000
Taux sarcasme train/val/test : 0.17276188789515629 0.17290836653386454 0.27366666666666667


In [5]:
for name, frame in [('train', train), ('validation', val), ('test', test)]:
    print('\n', name)
    print('Global :', frame['sarcasm'].astype(int).mean())
    print('Dialecte :')
    display(frame.groupby('dialect')['sarcasm'].apply(lambda x: x.astype(int).mean()).to_frame('taux_sarcasme'))
    print('Sentiment :')
    display(frame.groupby('sentiment')['sarcasm'].apply(lambda x: x.astype(int).mean()).to_frame('taux_sarcasme'))

always_zero = np.zeros(len(y_val), dtype=int)
print('Accuracy jamais sarcastique :', (always_zero == y_val).mean())
print('F1 sarcastique jamais sarcastique :', f1_score(y_val, always_zero, zero_division=0))



 train
Global : 0.17276188789515629
Dialecte :


,taux_sarcasme
dialect,
egypt,0.346170
gulf,0.248264
levant,0.214411
magreb,0.394737
msa,0.108512


Sentiment :


,taux_sarcasme
sentiment,
NEG,0.419634
NEU,0.029828
POS,0.026849



 validation
Global : 0.17290836653386454
Dialecte :


...


 test
Global : 0.27366666666666667
Dialecte :


,taux_sarcasme
dialect,
egypt,0.506536
gulf,0.177019
levant,0.297872
magreb,0.000000
msa,0.256134


Sentiment :


,taux_sarcasme
sentiment,
NEG,0.422779
NEU,0.141711
POS,0.010435


Accuracy jamais sarcastique : 0.8270916334661355
F1 sarcastique jamais sarcastique : 0.0


## A5. Baseline TF-IDF mots 1-2 et caracteres 2-5

Le seuil est choisi sur la validation en maximisant le F1 de la classe sarcastique.

In [6]:
word_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
word_model.fit(X_train, y_train)
word_scores = word_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.20, 0.81, 0.01)
best_threshold = max(thresholds, key=lambda t: f1_score(y_val, word_scores >= t, zero_division=0))
add_result('TF-IDF mots', y_val, word_scores, best_threshold)
print('Seuil validation :', best_threshold)
print(classification_report(y_val, word_scores >= best_threshold, target_names=['non sarcastique', 'sarcastique'], zero_division=0))


Seuil validation : 0.48000000000000026
                 precision    recall  f1-score   support

non sarcastique       0.91      0.79      0.85      1038
    sarcastique       0.38      0.63      0.48       217

       accuracy                           0.76      1255
      macro avg       0.65      0.71      0.66      1255
   weighted avg       0.82      0.76      0.78      1255



In [7]:
char_model = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
char_model.fit(X_train, y_train)
char_scores = char_model.predict_proba(X_val)[:, 1]
add_result('TF-IDF caracteres', y_val, char_scores, 0.5)


{'modele': 'TF-IDF caracteres',
 'split': 'validation',
 'F1 sarcastique': 0.5361216730038023,
 'PR-AUC': 0.5160995337452084,
 'Macro-F1': 0.7065689010180303,
 'seuil': 0.5}

## A6. Raccourcis et gestion du desequilibre

On compare ponderation, sur-echantillonnage et seuil. Le sur-echantillonnage est fait uniquement dans train.

In [8]:
def remove_cues(text):
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'[@#]\S+', ' TOKEN ', text)
    return re.sub(r'[!??]+', ' PUNCT ', text)

robust_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
robust_model.fit(X_train.map(remove_cues), y_train)
robust_scores = robust_model.predict_proba(X_val.map(remove_cues))[:, 1]
add_result('TF-IDF sans indices', y_val, robust_scores, best_threshold)

minority = train[train['sarcasm'] == True]
majority = train[train['sarcasm'] == False]
resampled = pd.concat([majority, minority.sample(len(majority), replace=True, random_state=RANDOM_STATE)])
resampled_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
resampled_model.fit(resampled['text_clean'], resampled['sarcasm'].astype(int))
resampled_scores = resampled_model.predict_proba(X_val)[:, 1]
add_result('TF-IDF sur-echantillonnage', y_val, resampled_scores, 0.5)

shortcut_features = pd.DataFrame({
    'url': X_train.str.count(r'https?://'), 'mention': X_train.str.count('@'),
    'hashtag': X_train.str.count('#'), 'longueur': X_train.str.len(),
    'questions': X_train.str.count(r'[??]'), 'exclamations': X_train.str.count('!')
})
shortcut_valid = pd.DataFrame({
    'url': X_val.str.count(r'https?://'), 'mention': X_val.str.count('@'),
    'hashtag': X_val.str.count('#'), 'longueur': X_val.str.len(),
    'questions': X_val.str.count(r'[??]'), 'exclamations': X_val.str.count('!')
})
shortcut_model = LogisticRegression(class_weight='balanced', max_iter=1000).fit(shortcut_features, y_train)
shortcut_scores = shortcut_model.predict_proba(shortcut_valid)[:, 1]
add_result('Raccourcis uniquement', y_val, shortcut_scores, 0.5)


{'modele': 'Raccourcis uniquement',
 'split': 'validation',
 'F1 sarcastique': 0.36847946725860153,
 'PR-AUC': 0.3238775144587555,
 'Macro-F1': 0.5074218343129552,
 'seuil': 0.5}